# Stages 3 and 4: the training loop, then a few epochs with logic checks

The loop is `trainer.fit`, the same function the script `train.py` calls. This notebook runs it for a few epochs and runs the checks that show the training is correct, not only that it ran.

Before you promote this to the script, use **Restart and Run All** so that no hidden state from earlier cells is hiding a bug. All checks must read PASS.

In [ ]:
# @title Find the project code (mounts Drive on a fresh Colab machine)
import glob, os
PROJECT = "colab-mnist-test"
DRIVE = "/content/drive"


def find_code_dirs():
    """Folders on this machine that hold the project code: the synced Drive project, or an uploaded copy."""
    places = [f"{DRIVE}/MyDrive/research/{PROJECT}", *sorted(glob.glob(f"{DRIVE}/Othercomputers/*/research/{PROJECT}")), f"/content/{PROJECT}"]
    return [d for d in places if os.path.isfile(os.path.join(d, "models.py"))]


if os.path.isdir("/content") and not find_code_dirs():
    from google.colab import drive
    drive.mount(DRIVE)  # a pop-up asks you to approve: click Connect to Google Drive within two minutes
CODE_DIRS = find_code_dirs()
print("project code found in:", CODE_DIRS or "nowhere on this machine (on this computer: the folder above the notebook)")

In [ ]:
# @title Setup and settings
import json, os, sys, time
sys.dont_write_bytecode = True  # keep __pycache__ out of the folder that Drive syncs
for path in ("..", *CODE_DIRS):  # project root on this computer, the synced Drive project or an uploaded copy on Colab
    if os.path.isfile(os.path.join(path, "models.py")) and os.path.abspath(path) not in sys.path:
        sys.path.insert(0, os.path.abspath(path))
import matplotlib.pyplot as plt
import torch
import augment, checks, data, models, paths, quicktrain, trainer

NOTEBOOK = "02_pipeline"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = paths.data_dir()
FIG_DIR = paths.figure_dir(NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Settings you can change ----
MODEL = "cnn"                                                      # "cnn" or "mlp"
AUGMENT = False                                                    # random shift and rotation on training images
EPOCHS = int(os.environ.get("MNIST_EPOCHS", "3" if ON_COLAB else "2"))
TRAIN_SIZE = 20000 if ON_COLAB else 2000                           # images used for training
VAL_SIZE = 5000 if ON_COLAB else 1000                              # images held out to measure progress
BATCH_SIZE = 64
LR = 1e-3
SEED = 0
print("device:", device, "| torch", torch.__version__, "| model:", MODEL, "| epochs:", EPOCHS)

In [ ]:
# @title Load the images and look at them
raw = __import__("torchvision").datasets.MNIST(DATA_DIR, train=True, download=True)
train_full, val_source, test_set = data.load_mnist(DATA_DIR, augment_train=AUGMENT)
train_set, val_set, train_idx, val_idx = data.make_sets(train_full, val_source, TRAIN_SIZE, VAL_SIZE, SEED)
print("training images:", len(train_set), "| validation images:", len(val_set))
fig, axes = plt.subplots(3, 8, figsize=(10, 4.2))
for ax, i in zip(axes.flat, train_idx[:24]):
    image, label = raw[i]
    ax.imshow(image, cmap="gray"); ax.set_title(str(label), fontsize=10); ax.axis("off")
fig.suptitle("The first 24 training images, with their labels")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "1_samples.png"), dpi=110); plt.show()

In [ ]:
# @title Checks before training (stage 4, part 1)
from torch.utils.data import DataLoader
torch.manual_seed(SEED)
probe = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
model = models.build_model(MODEL).to(device)
before = [
    checks.check_no_overlap(train_idx, val_idx),
    checks.check_initial_loss(model, probe, 10, device),
    checks.check_overfit_batch(lambda: models.build_model(MODEL), probe, device),
]
print(checks.format_results(before))
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# @title Train a few epochs with the shared loop
torch.manual_seed(SEED)
model = models.build_model(MODEL).to(device)          # a fresh model: the overfit check used its own copy
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
started = time.time()
history, state, _ = trainer.fit(model, optimizer, train_set, val_set, EPOCHS, BATCH_SIZE, device, seed=SEED)
print(f"trained {EPOCHS} epochs ({state.step} steps) in {time.time() - started:.1f} s")
print("epoch | train loss | train acc | val loss | val acc")
for k in range(EPOCHS):
    print(f"{k + 1:5d} | {history['train_loss'][k]:10.4f} | {history['train_acc'][k]:9.4f} | "
          f"{history['val_loss'][k]:8.4f} | {history['val_acc'][k]:7.4f}")

In [ ]:
# @title Checks after training (stage 4, part 2) and the verdict
after = [checks.check_finite(history), checks.check_loss_decreases(history), checks.check_above_chance(history, 10)]
print(checks.format_results(after))
verdict = checks.all_passed(before + after)
print()
print("ALL CHECKS PASSED: ready to promote this loop to the script (stage 5)." if verdict
      else "A CHECK FAILED: fix it here before running the script.")

In [ ]:
# @title Plot the loss and accuracy curves
steps_per_epoch = len(history["step_loss"]) // EPOCHS
window = max(1, len(history["step_loss"]) // 25)
smooth = [sum(history["step_loss"][max(0, i - window + 1):i + 1]) / len(history["step_loss"][max(0, i - window + 1):i + 1])
          for i in range(len(history["step_loss"]))]
epoch_steps = [steps_per_epoch * (k + 1) for k in range(EPOCHS)]
epochs = list(range(1, EPOCHS + 1))
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
ax_loss.plot(history["step_loss"], color="#cccccc", label="training loss, every batch")
ax_loss.plot(smooth, color="#b56a00", label="training loss, smoothed")
ax_loss.plot(epoch_steps, history["val_loss"], "o-", color="#132030", label="validation loss, each epoch")
ax_loss.set_xlabel("training step"); ax_loss.set_ylabel("loss"); ax_loss.set_title("Loss"); ax_loss.legend()
ax_acc.plot(epochs, history["train_acc"], "o-", color="#b56a00", label="training accuracy")
ax_acc.plot(epochs, history["val_acc"], "o-", color="#132030", label="validation accuracy")
ax_acc.set_xticks(epochs); ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("accuracy"); ax_acc.set_title("Accuracy"); ax_acc.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "3_curves.png"), dpi=110); plt.show()

In [ ]:
# @title Look at the predictions (wrong ones in red)
model.eval()
xs, ys = next(iter(DataLoader(val_set, batch_size=500)))
with torch.no_grad():
    guesses = model(xs.to(device)).argmax(1).cpu()
model.train()
wrong = int((guesses != ys).sum())
fig, axes = plt.subplots(3, 8, figsize=(10, 4.4))
for ax, image, truth, guess in zip(axes.flat, xs[:24], ys[:24], guesses[:24]):
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"{guess.item()} (is {truth.item()})", fontsize=9, color="green" if guess == truth else "red"); ax.axis("off")
fig.suptitle(f"Predicted digit and true digit; {wrong} of the first {len(ys)} validation images are wrong")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "4_predictions.png"), dpi=110); plt.show()

In [ ]:
# @title Save the history and the check results
out = os.path.join(FIG_DIR, "history.json")
with open(out, "w") as handle:
    json.dump({"model": MODEL, "epochs": EPOCHS, "train_size": TRAIN_SIZE, "val_size": VAL_SIZE, "lr": LR, "device": str(device),
               "all_checks_passed": verdict, "checks": [vars(r) for r in before + after], "history": history}, handle, indent=2)
print("saved", out)